# Assumption Ops: operational intervention comparison

Run **Runtime → Run all** in Google Colab. CPU only; no API keys or business accounts.

This notebook goes beyond validating the reasoning mechanism: it loads ordinary CSV files, records a supplier delay, compares three operational responses against a consistent baseline, exposes all cost components, and exports an advisory comparison.

The supplied data and action costs are illustrative. Scores are **weighted business penalty units**, not calibrated currency or measured ROI. Scenario ranking neither changes accepted facts nor approves or executes a candidate.

In [ ]:
from collections import defaultdict
import importlib.util
import subprocess
import sys

needs_install = importlib.util.find_spec("assumption_ops") is None
if not needs_install:
    import assumption_ops as installed_ops
    needs_install = not hasattr(installed_ops, "ScenarioComparison")
if needs_install:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
        "git+https://github.com/crypticsymmetry/improved-giggle.git",
    ])
    # Refresh only this framework if an older version was imported in this runtime.
    for module_name in list(sys.modules):
        if module_name == "assumption_ops" or module_name.startswith("assumption_ops."):
            del sys.modules[module_name]

from assumption_ops import (
    ATMS, OperationsPipeline, Supply, Order, Policy, Scenario,
    StaleDecisionError, evaluate_plan, load_csv_scenario, write_comparison_csv,
)
from assumption_ops.evidence import EvidenceConflict
from pprint import pprint


## 1. Load supply and order CSVs

These are the same fixture rows as the first notebook. To use your own data, replace the two paths passed to `load_csv_scenario`; the files must use the demonstrated headers. Quantities and relative days are nonnegative integers; substitutions are one-for-one.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from IPython.display import HTML, display
from html import escape


def show_table(rows, columns=None):
    columns = columns or list(rows[0])
    header = "".join(f"<th>{escape(str(column))}</th>" for column in columns)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(str(row[column]))}</td>" for column in columns) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>"))


supplies_csv = """id,sku,quantity,available_day,unit_cost
stock,A,3,0,1
shipment,A,7,2,1
substitute,B,4,0,3
"""
orders_csv = """id,sku,quantity,due_day,priority
urgent,A,5,3,3
standard,A,5,4,1
"""
with TemporaryDirectory() as folder:
    supply_path, order_path = Path(folder)/"supplies.csv", Path(folder)/"orders.csv"
    supply_path.write_text(supplies_csv, encoding="utf-8")
    order_path.write_text(orders_csv, encoding="utf-8")
    data = load_csv_scenario(supply_path, order_path, Policy(substitutions={"A": ("B",)}))

ops = OperationsPipeline()
ops.seed(data.supplies, data.orders, data.policy)
initial = ops.plan()
initial_evaluation = evaluate_plan(data.supplies, data.orders, data.policy, initial.plan)
show_table(initial_evaluation.orders)
show_table([initial_evaluation.to_dict()["costs"]])
assert initial_evaluation.metrics["unfilled_units"] == 0


## 2. Review a supplier delay

The incoming observation is a proposal. Accepting it exposes a disagreement with the original confirmation; it does not quietly overwrite history. The explicit resolution below models a reviewer selecting the revised supplier confirmation.

In [ ]:
delay_id = ops.propose("supply:shipment", "available_day", 8, "demo: revised supplier confirmation")
ops.accept(delay_id)
impact = ops.impact()
show_table(impact["conflicts"])
print("Affected orders:", impact["affected_order_ids"])
try:
    ops.plan(previous=initial.plan)
except EvidenceConflict:
    print("Planning correctly blocked until source conflict resolution.")
else:
    raise AssertionError("Conflicting evidence must block planning")
ops.resolve("supply:shipment", "available_day", delay_id)


## 3. Compare responses under the same scoring model

- **Baseline:** allocate from the accepted delayed-supply snapshot without an intervention.
- **Expedite shipment:** hypothesize that it can arrive by day 3; add a fixed action cost of 40.
- **Buy more substitutes:** hypothesize total substitute stock of 7 units; add a fixed action cost of 20. Per-unit acquisition costs remain included.
- **Permit late fulfillment:** relax deadline eligibility while keeping the same lateness penalty and other scoring weights.

Every solve uses the same prior allocation and unchanged shortage, substitution, lateness, and disruption weights. The comparison API rejects changes to order demand or priority and rejects altered penalty weights. Availability, policy permission, and action prices still need real confirmation before an operational decision.

In [ ]:
candidates = [
    Scenario("expedite shipment", {"supply:shipment": {"available_day": 3}}, action_cost=40),
    Scenario("buy more substitute stock", {"supply:substitute": {"quantity": 7}}, action_cost=20),
    Scenario("permit late fulfillment", {"policy": {"config": {"allow_late": True}}}),
]
evidence_before, operations_before, decisions_before = ops.store.events(), ops.events(), ops.decisions()
comparison = ops.compare_scenarios(candidates, previous=initial.plan)
assert ops.store.events() == evidence_before
assert ops.events() == operations_before
assert ops.decisions() == decisions_before
show_table(comparison.rows(), [
    "rank", "name", "total_score", "delta_vs_baseline", "unfilled_units", "late_units", "substitute_units",
])
print("Snapshot revisions:", comparison.evidence_revision, comparison.operations_revision)
assert comparison.baseline.evaluation.metrics["unfilled_units"] == 3
assert comparison.rows()[0]["name"] == "expedite shipment"
assert all(outcome.evaluation.metrics["unfilled_units"] == 0 for outcome in comparison.alternatives)


## 4. Inspect the tradeoffs and the proposed allocations

The baseline score includes a penalty for three unfilled standard-order units. Expediting restores the original allocation and adds the intervention fee. Buying additional substitutes changes the allocation; late fulfillment has an explicit service penalty.

The cost breakdown is independently reconstructed from the returned allocation. It must equal the optimizer's objective; `total_score` adds the separate intervention cost. Scenario plans remain advisory and have no persisted decision IDs.

In [ ]:
show_table(comparison.rows(), [
    "name", "acquisition", "lateness", "substitution", "shortage", "disruption", "action_cost", "total_score",
])
best = min((comparison.baseline, *comparison.alternatives), key=lambda outcome: (outcome.total_score, outcome.name))
show_table(best.evaluation.orders)
print("Best modeled response:", best.name)
print("Solver status:", best.plan.status)
print("No candidate has been accepted, approved, or dispatched.")
assert all(outcome.evaluation.costs.total == outcome.plan.objective for outcome in comparison.alternatives)
assert best.total_score == best.evaluation.costs.total + best.action_cost


## 5. Export and customize

The CSV includes ranked scores, service metrics, cost components, and solver status. String fields are escaped when needed to prevent spreadsheet formula evaluation. To change the experiment, edit the CSV inputs, policy, candidate overrides, or action costs and rerun.

In Colab, the file is available through the Files panel. You may uncomment the download lines below.

In [ ]:
report_path = Path("intervention_comparison.csv")
write_comparison_csv(comparison, report_path)
print("Wrote:", report_path.resolve())
print(report_path.read_text(encoding="utf-8"))
# from google.colab import files
# files.download(str(report_path))


## 6. Create a reviewed plan for the currently accepted facts

This produces a plan for the delayed shipment, not for an unconfirmed intervention. A later verified expedite confirmation could enter through `propose`, `accept`, and `resolve`, followed by a new plan. The illustrative fixed intervention fee is part of the advisory comparison, not a booked payment or an inventory reservation.

The original walkthrough demonstrates approval, stale-evidence rejection, local commit, and idempotent reservations. Here we leave every decision unapproved.

In [ ]:
reviewed_plan = ops.plan(previous=initial.plan)
assert reviewed_plan.plan.objective == comparison.baseline.plan.objective
assert all(decision.status == "proposed" for decision in reviewed_plan.decisions)
show_table(comparison.baseline.evaluation.orders)
print("Persisted proposed decisions:", len(reviewed_plan.decisions))
print("External API calls:", 0)
ops.close()
